In [1]:
import os
os.chdir("..")  # run from the repo root so "data/raw" resolves

from gpu_dev_assist.ingestion.chunking_embedding import ChunkingEmbeddingPipeline
from gpu_dev_assist.ingestion.data_loader import load_all_documents

c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
pipe = ChunkingEmbeddingPipeline(model_name='all-MiniLM-L6-v2')
chunks = pipe.chunk_documents(load_all_documents("data/raw"))

tok = pipe.model.tokenizer
lengths = sorted(len(tok(c.page_content)["input_ids"]) for c in chunks)
limit = pipe.model.max_seq_length
over = sum(l > limit for l in lengths)

print(f"limit: {limit} tokens")
print(f"median: {lengths[len(lengths) // 2]}, max: {lengths[-1]}")
print(f"over limit: {over} of {len(chunks)} ({100 * over / len(chunks):.0f}%)")

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2710.01it/s]


Found 2 PDF files to process

Processing: oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


  ✓ Loaded 405 pages

Processing: oneapi_programming-guide_2025.1-771723-848694.pdf
  ✓ Loaded 129 pages

Total documents loaded: 534
limit: 256 tokens
median: 238, max: 524
over limit: 664 of 1585 (42%)


In [3]:
pipe = ChunkingEmbeddingPipeline(model_name='BAAI/bge-small-en-v1.5')
chunks = pipe.chunk_documents(load_all_documents("data/raw"))

tok = pipe.model.tokenizer
lengths = sorted(len(tok(c.page_content)["input_ids"]) for c in chunks)
limit = pipe.model.max_seq_length
over = sum(l > limit for l in lengths)

print(f"limit: {limit} tokens")
print(f"median: {lengths[len(lengths) // 2]}, max: {lengths[-1]}")
print(f"over limit: {over} of {len(chunks)} ({100 * over / len(chunks):.0f}%)")


c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aishv\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\hugg

Found 2 PDF files to process

Processing: oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf
  ✓ Loaded 405 pages

Processing: oneapi_programming-guide_2025.1-771723-848694.pdf
  ✓ Loaded 129 pages

Total documents loaded: 534


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors


limit: 512 tokens
median: 238, max: 524
over limit: 2 of 1585 (0%)


### Debug the retrieval results for the below query, trying to understand why the llm didn't answer correctly

In [ ]:
from gpu_dev_assist.ingestion.chunking_embedding import ChunkingEmbeddingPipeline
from gpu_dev_assist.retrieval.dense import Retriever
from gpu_dev_assist.ingestion.vector_store import VectorStore
import os

os.environ["HF_HUB_OFFLINE"] = "1"

vs = VectorStore()
print(f"Collection count: {vs.collection.count()}")      # must be 1585
retriever = Retriever(vs, ChunkingEmbeddingPipeline())
results = retriever.query_retrieve("When should I use shared local memory in a SYCL kernel?", top_k=10)
# 1. What does retrieval return for this question?
for rank, r in enumerate(
    results, start=1
):
    print(rank, r["source"][:20], r["page"], r["distance"], r["text"][:70].replace("\n", " "))

# 2. Which pages actually contain the phrase?
for phrase in ["Shared Local Memory", "shared local memory", "SLM"]:
    hits = vs.collection.get(where_document={"$contains": phrase}, include=["metadatas"])
    print(phrase, sorted({m["page"] + 1 for m in hits["metadatas"]}))

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'BAAI/bge-small-en-v1.5'.


Collection count: 1585


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.04it/s]

1 oneapi_optimization- 142 0.16 • Avoiding Moving Data Back and Forth between Host and Device • Optimi
2 oneapi_optimization- 128 0.183 auto A = sycl::malloc_shared<float *>(M, sycl_device, sycl_context);  
3 oneapi_optimization- 106 0.185 with shared context. This will prevent the need to transfer the input 
4 oneapi_optimization- 128 0.187 However, this naive implementation won’t give the best possible perfor
5 oneapi_optimization- 143 0.197 between them is that shared allocations are free to migrate between ho
6 oneapi_optimization- 83 0.197 std::cout << "Reduction Sum : " << h_a[0] << "\n";   std::cout       <
7 oneapi_optimization- 145 0.201 Shared Allocations Next we show the same algorithm implemented with sh
8 oneapi_optimization- 327 0.205 • Avoid stride-1 memory accesses in slowest moving dimension for 2D an
9 oneapi_optimization- 38 0.209 int *data = sycl::malloc_shared<int>(N, q);   int *data2 = sycl::mallo
10 oneapi_optimization- 142 0.21 shared allocated shared between ho

### Trying with BM25 retrieval method 

In [11]:
import re
from rank_bm25 import BM25Okapi

data = vs.collection.get(include=["documents", "metadatas"])
docs, metas = data["documents"], data["metadatas"]


def tokenize(text):
    return re.findall(r"[a-z0-9_]+", text.lower())


bm25 = BM25Okapi([tokenize(d) for d in docs])

query = "When should I use shared local memory in a SYCL kernel?"
scores = bm25.get_scores(tokenize(query))
top = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:10]

for rank, i in enumerate(top, start=1):
    print(rank, metas[i]["page"] + 1, round(scores[i], 2), docs[i][:70].replace("\n", " "))

1 98 17.76 SYCL also supports built-in reduction operations, and you should use i
2 16 16.74 Once the problem areas are identified, utilize the tools described in 
3 327 16.68 • Avoid stride-1 memory accesses in slowest moving dimension for 2D an
4 395 16.47 EU Active is the percentage of time when ALU0 or ALU1 were executing s
5 72 16.25 int t = 0;             for (int j = 0, k = local_id; j < _M; ++j, ++k)
6 57 16.09 ``x = sg.load(global_ptr(&(data2[base + 0]))); sg.store(global_ptr(&(d
7 280 15.85 2.4 atomic Directive with SLM Building on the OpenMP atomic directive,
8 151 15.65 h.parallel_for(num_items, [=](auto i) {         int t = a_acc[i] + b_a
9 151 15.59 Unified Shared Memory (USM), instead of system memory (such as memory 
10 64 15.57 v4.WriteBack(vals_out_offs + 72);                       });      }).wa
